# Grasp Method Comparison — full runs, auto-calibrated

**Flow:** restart kernel → run **c01–c04** + **cmp_setup** + **cmp_calib** (setup; auto-calibrates
the camera) → run **1 / 2 / 3 / 4** (one full grasp per method, reset the object between) → **cmp_table**.

Each method runs the SAME execution code (`grasp_runner.py`); only the planner differs.
Learned methods need their bridge up (`scripts/grasp_detectors/INSTALL.md`); ones that are
down are skipped. `pca_baseline` works today.

In [ ]:
import subprocess, os, time

SAM3_PY  = os.path.expanduser('~/sam3_env/bin/python3')
SAM3_SCR = os.path.expanduser('~/magpie_control/scripts/sam3_infer.py')
ROS      = 'source /opt/ros/humble/setup.bash && source ~/ws_ctrl/install/setup.bash'

def ros_proc(cmd, log):
    open(log, 'w').close()
    return subprocess.Popen(f'{ROS} && {cmd}', shell=True, executable='/bin/bash',
                            stdout=open(log, 'a'), stderr=subprocess.STDOUT)

print('Killing stale processes...')
for p in ['ur5_node', 'gripper_node', 'realsense2_camera', 'sam3_infer', 'slip_guard_node', 'ft_sensor_node']:
    subprocess.run(f'pkill -9 -f {p}', shell=True)
subprocess.run('rm -f /tmp/sam3.sock', shell=True)
time.sleep(3)

PROCS = {}
PROCS['ur5']     = ros_proc('ros2 run magpie_control ur5_node',     '/tmp/log_ur5.txt')
PROCS['gripper'] = ros_proc('ros2 run magpie_control gripper_node', '/tmp/log_gripper.txt')
PROCS['ft']      = ros_proc('ros2 run magpie_control ft_sensor_node', '/tmp/log_ft.txt')
PROCS['camera']  = ros_proc(
    'ros2 run realsense2_camera realsense2_camera_node --ros-args -r __ns:=/camera/gripper_camera',
    '/tmp/log_camera.txt')
PROCS['slip_guard'] = ros_proc(
    f'python3 {os.path.expanduser("~/magpie_control/scripts/slip_guard_node.py")}',
    '/tmp/log_slip_guard.txt')
PROCS['sam3']    = subprocess.Popen(
    [SAM3_PY, SAM3_SCR, '--socket'],
    stdout=open('/tmp/log_sam3.txt', 'w'), stderr=subprocess.STDOUT)

# ── Grasp memory (force priors + DINO RAG) ──────────────────────────
import sys as _sys
_sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts'))
from grasp_memory import GraspMemory
gm = GraspMemory(os.path.expanduser('~/magpie_control/data/grasp_log'))
print(f'GraspMemory loaded — {len(gm._priors)} known objects')

print('Waiting for SAM3 (up to 90s)...')
for i in range(90):
    time.sleep(1)
    if os.path.exists('/tmp/sam3.sock'):
        print(f'  SAM3 ready after {i+1}s')
        break
    if PROCS['sam3'].poll() is not None:
        print('  SAM3 crashed:'); print(open('/tmp/log_sam3.txt').read()[-600:])
        break
    if (i+1) % 20 == 0:
        print(f'  {i+1}s...')

time.sleep(3)
print()
for n, p in PROCS.items():
    status = 'OK' if p.poll() is None else f'EXITED  -> tail /tmp/log_{n}.txt'
    print(f'  {n:<10} {status}')

In [ ]:
# Load ROS shared libraries so rclpy imports in any VS Code kernel
import sys, os, ctypes, glob

for _d in [
    '/opt/ros/humble/lib/x86_64-linux-gnu',
    '/opt/ros/humble/lib',
    '/home/user/ws_ctrl/install/magpie_msgs/lib',
    '/home/user/ws_ctrl/install/magpie_control/lib',
]:
    for _so in sorted(glob.glob(_d + '/*.so*')):
        try:
            ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

for _p in [
    '/opt/ros/humble/local/lib/python3.10/dist-packages',
    '/opt/ros/humble/lib/python3.10/site-packages',
    '/home/user/ws_ctrl/install/magpie_msgs/local/lib/python3.10/dist-packages',
    '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages',
    '/home/user/.local/lib/python3.10/site-packages',
    '/home/user/magpie_control/src',
    '/home/user/magpie_control/scripts',
]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import rclpy
print('rclpy OK')

In [ ]:
import time, re, json, tempfile, base64, socket as _sock
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import cv2
%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi']     = 80   # keeps output file size small for GitHub

from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy
from sensor_msgs.msg import Image as RosImage, CameraInfo
from geometry_msgs.msg import PoseStamped, Pose, WrenchStamped
from std_srvs.srv import Trigger
from cv_bridge import CvBridge
from magpie_msgs.msg import GripperState
from magpie_msgs.srv import MoveLinear, SetGripperForce, SetGripperPosition
from magpie_control import poses
from magpie_control.homog_utils import homog_xform, R_krot
from magpie_control.gripper_arc import fingertip_drop
from google import genai
from google.genai import types as gtypes
from pointcloud_utils import build_segmented_pcd, denoise_pcd, analyse_pcd

# --- CONFIG ---
OBJECT         = ''   # blank = Gemini auto-detects
SAM3_SOCK      = '/tmp/sam3.sock'
GEMINI_KEY     = os.environ.get('GEMINI_API_KEY', '')
GRIPPER_LEN    = 0.231
HARD_FLOOR_Z   = 0.025
TABLE_Z        = None   # set by running the measure cell below
APPROACH_H     = 0.10
# Camera extrinsic — camera optical frame -> TCP frame.
# The camera is clocked -90 deg about the tool axis. This is a PROPER rotation
# (det +1). An earlier fix used Rz(90)@diag(-1,1,1), a REFLECTION (det -1): that
# corrected world Y but flipped world X (non-physical handedness). Rz(-90) flips
# BOTH lateral axes consistently, which is the real mounting.
# If a single axis still looks reversed, the true clocking is one of Rz(0/90/180/270)
# — run the xy_calib cell to measure it instead of guessing.
_TCP_TO_CAM    = homog_xform(R_krot([0, 0, 1], -np.pi/2), [0, 0, 0.120])

if not GEMINI_KEY:
    print('WARNING: GEMINI_API_KEY not set.')
print('Imports OK')

In [ ]:
# Helper: quaternion <-> rotation vector
def _quat_to_rv(w, x, y, z):
    a = 2.0 * np.arccos(np.clip(w, -1, 1))
    s = np.sin(a / 2)
    return np.zeros(3) if s < 1e-10 else a * np.array([x, y, z]) / s

def _rv_to_quat(rv):
    a = np.linalg.norm(rv)
    if a < 1e-10:
        return (1., 0., 0., 0.)
    ax = rv / a
    return (np.cos(a/2), ax[0]*np.sin(a/2), ax[1]*np.sin(a/2), ax[2]*np.sin(a/2))

def _mat_to_pose(mat):
    v = poses.pose_mtrx_to_vec(np.array(mat))
    w, x, y, z = _rv_to_quat(np.array(v[3:]))
    p = Pose()
    p.position.x, p.position.y, p.position.z = v[0], v[1], v[2]
    p.orientation.w, p.orientation.x = w, x
    p.orientation.y, p.orientation.z = y, z
    return p

# QoS profiles matching RealSense D405 publisher settings
img_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.TRANSIENT_LOCAL)
inf_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.VOLATILE)

class Demo(Node):
    def __init__(self):
        super().__init__('magpie_demo')
        self.bridge = CvBridge()
        self.color = self.depth = self.caminfo = self.tcp = self.gs = None
        self.wrench = None   # WrenchStamped from OptoForce FT (may be None)
        NS = '/camera/gripper_camera/camera'
        self.create_subscription(
            RosImage,    NS+'/color/image_raw',
            lambda m: setattr(self, 'color', self.bridge.imgmsg_to_cv2(m, 'rgb8')), img_qos)
        self.create_subscription(
            RosImage,    NS+'/depth/image_rect_raw',
            lambda m: setattr(self, 'depth', self.bridge.imgmsg_to_cv2(m, 'passthrough')), img_qos)
        self.create_subscription(
            CameraInfo,  NS+'/color/camera_info',
            lambda m: setattr(self, 'caminfo', m), inf_qos)
        self.create_subscription(
            GripperState, '/gripper/state',
            lambda m: setattr(self, 'gs', m), 1)
        self.create_subscription(
            WrenchStamped, 'ft_sensor/wrench',
            lambda m: setattr(self, 'wrench', m), 10)
        self.create_subscription(
            PoseStamped, '/arm/tcp_pose',
            self._tcp_cb, 1)
        self.mv  = self.create_client(MoveLinear,         '/arm/move_l')
        self.tch = self.create_client(Trigger,             '/arm/teach_mode')
        self.opn = self.create_client(Trigger,             '/gripper/open')
        self.cls = self.create_client(Trigger,             '/gripper/close')
        self.frc = self.create_client(SetGripperForce,     '/gripper/set_force')
        self.pos = self.create_client(SetGripperPosition,  '/gripper/set_position')
        self.clr = self.create_client(Trigger,             '/gripper/clear_error')

    def _tcp_cb(self, m):
        p = m.pose
        rv = _quat_to_rv(p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z)
        self.tcp = poses.pose_vec_to_mtrx([p.position.x, p.position.y, p.position.z, *rv])

    def spin(self, n=10, t=0.15):
        for _ in range(n):
            rclpy.spin_once(self, timeout_sec=t)

    def _call(self, client, req, timeout=30.):
        client.wait_for_service(timeout_sec=4.)
        fut = client.call_async(req)
        rclpy.spin_until_future_complete(self, fut, timeout_sec=timeout)
        return fut.result()

    def move(self, mat, spd=0.08, acc=0.2):
        r = MoveLinear.Request()
        r.target_pose = _mat_to_pose(mat)
        r.speed = spd; r.acceleration = acc; r.async_mode = False
        resp = self._call(self.mv, r)
        if not resp.success:
            raise RuntimeError(resp.message)

    def open_g(self):  return self._call(self.opn, Trigger.Request())
    def close_g(self): return self._call(self.cls, Trigger.Request())

    def set_force(self, n):
        r = SetGripperForce.Request(); r.max_force = float(n)
        return self._call(self.frc, r)

    def set_pos(self, mm):
        r = SetGripperPosition.Request(); r.position = float(max(0, mm))
        return self._call(self.pos, r)

    def clear_err(self):
        # Re-enable AX-12 torque after overload shutdown (reset_packet_overload).
        # NOTE: this resets force limit to 2N, so call set_force() again after.
        if not self.clr.wait_for_service(timeout_sec=1.):
            return None
        return self._call(self.clr, Trigger.Request())

    def slip_guard_enable(self, force_n, slip_thresh, obj_u=None, obj_v=None, force_step=1.0):
        """Send config (force, thresh, object pixel coords, reclamp step) then enable the SlipGuardNode."""
        if not hasattr(self, '_sg_en'):
            from std_msgs.msg import Float32MultiArray
            self._sg_cfg_pub = self.create_publisher(
                Float32MultiArray, 'slip_guard/config', 1)
            self._sg_en  = self.create_client(Trigger, 'slip_guard/enable')
            self._sg_dis = self.create_client(Trigger, 'slip_guard/disable')
            from std_msgs.msg import String as _SgStr
            self._sg_events = []
            self.create_subscription(_SgStr, '/slip_guard/events',
                lambda msg: self._sg_events.append(msg.data), 10)
        from std_msgs.msg import Float32MultiArray
        cfg = Float32MultiArray()
        # [force, thresh, obj_u, obj_v] — pixel coords tell guard where to sample depth
        cfg.data = [float(force_n), float(slip_thresh),
                    float(obj_u) if obj_u is not None else -1.,
                    float(obj_v) if obj_v is not None else -1.,
                    float(force_step)]
        self._sg_events = []  # reset for this grasp — fresh DAgger log
        self._sg_cfg_pub.publish(cfg)
        time.sleep(0.1)
        if not self._sg_en.wait_for_service(timeout_sec=1.):
            print('[slip_guard] node not running — guard skipped')
            return
        self._call(self._sg_en, Trigger.Request())

    def slip_guard_disable(self):
        if hasattr(self, '_sg_dis') and self._sg_dis.wait_for_service(timeout_sec=0.5):
            self._call(self._sg_dis, Trigger.Request())

    def unteach(self):
        if not self.tch.wait_for_service(timeout_sec=2.):
            return
        r = self._call(self.tch, Trigger.Request())
        if r and 'enabled' in r.message.lower():
            self._call(self.tch, Trigger.Request())

    def wait_sensors(self, timeout=20.):
        t0 = time.time()
        while time.time() - t0 < timeout:
            rclpy.spin_once(self, timeout_sec=0.15)
            if all(v is not None for v in [self.color, self.depth, self.caminfo, self.tcp]):
                return True
        return False

try:
    rclpy.init()
except RuntimeError:
    pass

try:
    node.destroy_node()
except Exception:
    pass

node = Demo()
ok   = node.wait_sensors(20.)
print('Sensors ready:', ok,
      '| color:', node.color is not None,
      '| depth:', node.depth is not None,
      '| tcp:',   node.tcp   is not None)

In [ ]:
# ── Memory Manager ────────────────────────────────────────────────────────────
# Run this to inspect, rename, or delete entries from grasp memory.
#
# Override auto-detection for next pickup:
#   ACTIVE_OVERRIDE = 'measuring tape'
# Clear it after:
#   ACTIVE_OVERRIDE = ''

ACTIVE_OVERRIDE = ''   # ← set this before running 17ea9af9 if detection is wrong

print('=== Grasp Memory Contents ===')
if 'gm' not in dir():
    import sys; sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts'))
    from grasp_memory import GraspMemory
    gm = GraspMemory(os.path.expanduser('~/magpie_control/data/grasp_log'))
for _name, _p in sorted(gm._priors.items(), key=lambda x: -x[1]['n']):
    print(f'  {_name:<30} n={_p["n"]:>3}  force={_p["x"]:.1f}N  std={_p["P"]**0.5:.2f}N')
print(f'\nTotal: {len(gm._priors)} objects')

# ── To delete a wrong entry ────────────────────────────────────────────────
# del gm._priors['wrong name']; gm._save_priors(); print('Deleted.')

# ── To rename a wrong entry ───────────────────────────────────────────────
# gm._priors['correct name'] = gm._priors.pop('wrong name')
# gm._save_priors(); print('Renamed.')

# ── To reset a single object to re-learn from scratch ─────────────────────
# del gm._priors['measuring tape']; gm._save_priors(); print('Reset.')


In [ ]:
# ── Comparison setup: detectors, shared runner, SAM3 helper, run_method() ─────
import sys, socket as _sock, json as _json, tempfile as _tmpf, base64 as _b64
sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts'))
sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts/grasp_detectors'))
import rclpy, importlib
import grasp_runner, auto_calib
import grasp_detectors.pca_baseline, grasp_detectors.socket_detector
for _m in (grasp_runner, auto_calib, grasp_detectors.pca_baseline, grasp_detectors.socket_detector):
    importlib.reload(_m)   # pick up script edits without a kernel restart
from grasp_detectors.pca_baseline import PCABaseline
from grasp_detectors import socket_detector as _sd
from grasp_runner import capture_scene, full_run
from auto_calib import auto_calibrate
from google import genai as _genai
if 'gc' not in dir():
    gc = _genai.Client(api_key=os.environ.get('GEMINI_API_KEY', ''))

def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = _tmpf.mktemp(suffix='.jpg')
    cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path)
            s.sendall((_json.dumps({'image': tmp, 'query': query}) + '\n').encode())
            raw = b''
            while True:
                ch = s.recv(65536)
                if not ch: break
                raw += ch
        d = _json.loads(raw.decode().strip())
        if 'error' in d: raise RuntimeError(d['error'])
        boxes = np.array(d['boxes'], float); scores = np.array(d['scores'], float)
        mask = None
        if d.get('mask_b64') and len(boxes) > 0:
            h, w = d['mask_shape']
            mask = np.frombuffer(_b64.b64decode(d['mask_b64']), np.uint8).reshape(h, w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

CONST = dict(APPROACH_H=APPROACH_H, GRIPPER_LEN=GRIPPER_LEN,
             HARD_FLOOR_Z=HARD_FLOOR_Z, TABLE_Z=TABLE_Z)
DETECTORS = [PCABaseline(), _sd.contact_graspnet(), _sd.anygrasp(), _sd.graspgen()]
RESULTS = []

def run_method(name, force=4.0):
    det = {d.name: d for d in DETECTORS}.get(name)
    if det is None or not det.available():
        print(f'{name}: bridge not running — skipped (see grasp_detectors/INSTALL.md)'); return
    print(f'>>> {name}: full grasp on "{ACTIVE}" ...')
    scene = capture_scene(node, sam3_query, ACTIVE, tcp_to_cam=_TCP_TO_CAM)
    row = full_run(node, det, scene, CONST, rclpy, force=force)
    RESULTS.append(row); print('   ', row)
    print('    -> reset the object to the same spot before the next method')

print('methods:', [(d.name, 'up' if d.available() else 'no bridge') for d in DETECTORS])

In [ ]:
# ── AUTO-CALIBRATE: detect object + measure world->image mapping + set _TCP_TO_CAM ──
from google import genai as _genai
from google.genai import types as gtypes
gc = _genai.Client(api_key=os.environ.get('GEMINI_API_KEY', ''))
node.spin(6)
_, _b = cv2.imencode('.jpg', cv2.cvtColor(node.color.copy(), cv2.COLOR_RGB2BGR))
_r = gc.models.generate_content(model='gemini-2.5-flash', contents=[
    gtypes.Part.from_bytes(data=_b.tobytes(), mime_type='image/jpeg'),
    'What is the main graspable object? Reply 2-4 words only, no punctuation.'])
ACTIVE = ' '.join(_r.text.strip().lower().split()[:4])
if 'ACTIVE_OVERRIDE' in dir() and ACTIVE_OVERRIDE.strip(): ACTIVE = ACTIVE_OVERRIDE.strip().lower()
print('object:', ACTIVE)
_TCP_TO_CAM, _cal = auto_calibrate(node, sam3_query, ACTIVE)
print('AUTO-CALIBRATED ->', _cal)

In [ ]:
# 1. baseline (current pipeline)
run_method('pca_baseline')

In [ ]:
# 2. Contact-GraspNet (needs bridge)
run_method('contact_graspnet')

In [ ]:
# 3. AnyGrasp (needs bridge + license)
run_method('anygrasp')

In [ ]:
# 4. GraspGen (needs bridge)
run_method('graspgen')

In [ ]:
# ── Compare all full runs → pick a winner ────────────────────────────────────
import pandas as pd
if not RESULTS:
    print('No runs yet — run cells 1–4 first.')
else:
    df = pd.DataFrame(RESULTS)
    cols = [c for c in ['detector','held','grasp_force','final_aperture','angle_deg',
                        'plan_latency_ms','exec_time_s','score'] if c in df.columns]
    print(df[cols].to_string(index=False))
    if 'held' in df.columns:
        rate = df.groupby('detector')['held'].mean().sort_values(ascending=False)
        print('\nheld-rate:'); print(rate.to_string())
        print(f'\n=> suggested main method: {rate.index[0]}')